# Use note glide features to analyse scale structure of each singer

First read in the column information

In [ ]:
from pathlib import Path
import pandas as pd
import mir_eval
from IPython.display import Audio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from scipy.stats import multivariate_normal

# Choose the condition
condition = 'IM' # IM of SF
datadir = Path(f'C:/Users/monik/Documents/Paris/Internship/Data-Collection/Extracted-Data/Glides-Notes/{condition}')

# List of participants
participants = ['292711', '783520', '957921', 'djtosh', 'fosmvm', 'fubqui', 
                'fxdrni', 'glbbwi', 'kdkycn', 'lfraeu', 'nefwgv', 'nqqivt',
                'onbana', 'sdwmkv', 'uypoyr', 'vroghc', 'wxbeka', 'yenswd',
                'ytzemq']

# Read in file based on subject and songtype
subject = 'sdwmkv'
songtype = 'IMPROV' # IMPROV or Song-Forms
csvfile = next(datadir.glob(f'note-glide_data*{subject}*{songtype}.csv'))
print(csvfile)
df = pd.read_csv(csvfile)
df.columns

Make sure what you read in sounds some what like music

In [ ]:
sr = 22050
# Randomly choose a song
song_id = np.random.randint(1, 31)
songclip = df[df['SONG_ID'] == song_id]
# turn nans to -1 for sonify
songclip['FUNDAMENTAL_FREQUENCY'].iloc[songclip['FUNDAMENTAL_FREQUENCY'].isna()] = -1
songclip['FUNDAMENTAL_FREQUENCY'].iloc[songclip['NG_type'] == 'glide'] = -1
# play song
y_dig = mir_eval.sonify.pitch_contour(songclip['TIME_FRAME'], songclip['FUNDAMENTAL_FREQUENCY'], sr, length=30*sr)
print(f'Playing song {song_id}')
Audio(y_dig, rate=sr)

## Start with timing analysis.

Plotting the ratio of durations against each other.

In [ ]:
# Grab only note information
all_notes = df[df['NG_type'] == 'note']
note_durs = list()
# for each song
for s_id, songnotes in all_notes.groupby("SONG_ID"):
    # grab the first time point of the note to know when it started
    note_onsets = [x['TIME_FRAME'].iloc[0] for _, x in songnotes.groupby(['NG_ID'])]
    # save the note durations as the difference in time between notes (maybe should remove long pauses)
    note_dur = np.diff(note_onsets)
    note_dur = note_dur[note_dur >= .2]
    note_durs.append(note_dur)
plt.hist(np.hstack(note_durs), 30, weights=np.hstack(note_durs))

In [ ]:
# plot ratio of neighboring durations against ratio of durations 2 apart
for note_dur in note_durs:
    # get duration ratios
    note_ratio1 = note_dur[1:] / note_dur[:-1]
    note_ratio2 = note_dur[2:] / note_dur[:-2]
    # keep ratios between 0 and 1, ignoring order of longer vs shorter intervals
    note_ratio1[note_ratio1 > 1] = 1./note_ratio1[note_ratio1 > 1]
    note_ratio2[note_ratio2 > 1] = 1./note_ratio2[note_ratio2 > 1]
    # plot them all and see if something reveals itself
    plt.scatter(note_ratio1[:-1], note_ratio2, color='k', alpha=.5)
    

### Use Jacoby method

the above might have worked but it's pretty hard to see anything let's apply the method for Jacoby's tapping study

In [ ]:
# sections of 3 intervals
segns = 3
# projection matrix to convert 3 intervals into two-d space
M = np.array([(0, 0), (1, 0), (1/2, np.sqrt(3/4))])
# find the limits of this space
lims = np.eye(3)
rh_lims = lims @ M
print(rh_lims)
# plot the limits of the space as a triangle
fig, ax = plt.subplots()
ax.add_patch(Polygon(rh_lims, facecolor='k'))
# save durations in this normalized space
all_durs = list()
all_segn_durs = list()
for note_dur in note_durs:
    norm_durs = list()
    # for each 3 duration string
    for start in range(len(note_dur) - segns):
        finish = start + segns
        if np.any(np.logical_or(note_dur[start:finish] > 1.5, note_dur[start:finish] < .15)):
            continue
        # save the durations as a percentage of the time they take up
        norm_durs.append(note_dur[start:finish] / np.sum(note_dur[start:finish]))
        all_segn_durs.append(np.sum(note_dur[start]/np.sum(note_dur[start+1:finish])))
    all_durs.append(norm_durs)
    # project this normalized durations onto the triangle
    rh_sym = np.array(norm_durs) @ M
    # plot the scatter
    ax.scatter(*rh_sym.T, color='r', alpha=.1)
# show where the integer ratios are
anchors = np.array([(1, 1, 1,), (2, 1, 1), (1, 2, 1), (1, 1, 2), (2, 2, 1), (2, 1, 2), (1, 2, 2)], dtype=float)
anchors /= np.sum(anchors, axis=1, keepdims=True)
ach_sym = anchors @ M
plt.scatter(*ach_sym.T, marker='+', color='w')
ax.axis('off')


In [ ]:
plt.hist(all_segn_durs, 30)

## Above is a bit blurry but maybe people are varying across clips so lets look at individual clips

In [ ]:
# do the same but for each clip
anchors = np.array([(1, 1, 1,), (2, 1, 1), (1, 2, 1), (1, 1, 2), (2, 2, 1), (2, 1, 2), (1, 2, 2)], dtype=float)
anchors /= np.sum(anchors, axis=1, keepdims=True)
ach_sym = anchors @ M
fig, axs = plt.subplots(nrows=6, ncols=5, figsize=(16, 13))
for ax, norm_durs in zip(axs.flat, all_durs):
    ax.add_patch(Polygon(rh_lims, facecolor='k'))
    rh_sym = np.array(norm_durs) @ M
    ax.scatter(*rh_sym.T, s=8, color='r', alpha=.4)
    ax.axis('off')
    ax.scatter(*ach_sym.T, marker='+', color='w', alpha=.6)

### Estimating the primary durations

Here we have to deviate from Jacoby a bit.
They had simple repetitions of sounds and were able to use that to fit single gaussians on each iteration.
Our data is much more free form.
I am fitting each clip as a mixture of gaussians to see which fits are most common. 
Then afterwards we can combine gaussians across clips as they did

In [ ]:
from sklearn.mixture import GaussianMixture
# trynig to undersand how to partition each rhythm to see what was output
norm_durs = all_durs[18]
x = np.linspace(0.001, 1, 31)
x, y, z = np.meshgrid(x, x, x)
xyz = np.vstack([x.ravel(), y.ravel(), z.ravel()]).T
xyz = xyz / np.sum(xyz, axis=1, keepdims=True)
n_comps = 2
mixt = GaussianMixture(n_components=n_comps, covariance_type='tied', n_init=200, reg_covar=1e-6)
mixt.fit(norm_durs)
score = np.exp(mixt.score_samples(xyz))
xy_sym = xyz @ M
fig, ax = plt.subplots()
ax.add_patch(Polygon(rh_lims, facecolor='k'))
ax.scatter(xy_sym[:, 0], xy_sym[:, 1], s=5, c=score )
ax.axis('off')

mean_sym = mixt.means_ @ M
ax.scatter(*mean_sym.T, color='r', marker='+')
ax.scatter(*ach_sym.T, marker='+', color='w', alpha=.6)

### But plotting images with a scatter function is super heavy we need to fix this by plotting the gaussians we fit as images a la jacoby

First we need a helper function to get rid of the data outside of the triangle

In [ ]:
import numpy as np
import math

def is_outside_triangle(points, M):
    # Coordinates of the first two vertices
    vertex1 = M[0]
    vertex2 = M[1]
    vertex3 = M[2]


    # Calculate the areas of the three triangles formed by the point and the triangle's vertices
    triangle_area = abs(np.cross(vertex2 - vertex1, vertex3 - vertex1)) / 2
    sub_triangle1_area = abs(np.cross(points - vertex2, vertex3 - vertex2)) / 2
    sub_triangle2_area = abs(np.cross(vertex1 - points, vertex3 - vertex1)) / 2
    sub_triangle3_area = abs(np.cross(vertex1 - vertex2, points - vertex1)) / 2

    # Check if the sum of the sub-triangle areas is equal to the triangle area
    return np.abs(sub_triangle1_area + sub_triangle2_area + sub_triangle3_area - triangle_area) > 1e-6

# Test the function with some points
points = np.array([[0.5, 0.5], [0.2, 0.1], [-0.5, 0.5], [1.5, 0.5]])
result = is_outside_triangle(points, M)
print(result)

Then we plot the results across multivariate gaussians

In [ ]:
if mixt.covariance_type == 'tied':
    cov_sym = [M.T @ mixt.covariances_ @ M for _ in mixt.means_]
elif mixt.covariance_type == 'full':
    cov_sym = [M.T @ cov @ M for cov in mixt.covariances_]
elif mixt.covariance_type == 'spherical':
    cov_sym = [M.T @ (np.eye(mixt.means_.shape[1])*cov) @ M for cov in mixt.covariances_]
elif mixt.covariance_type == 'diag':
    cov_sym = [M.T @ np.diag(cov) @ M for cov in mixt.covariances_]

x = y =  np.linspace(0.001, 1, 101)
xG, yG = np.meshgrid(x, y)
xy = np.dstack([xG, yG])
ps = list()
for mu, cov in zip(mean_sym, cov_sym):
    ps.append(multivariate_normal.pdf(xy, mu, cov))
prob = np.mean(ps, axis=0)
prob[is_outside_triangle(xy, M)] = np.nan
fig, ax = plt.subplots()
plt.pcolormesh(x, x, prob, cmap=plt.cm.Reds)
ax.add_patch(Polygon(rh_lims, facecolor='none', edgecolor='k', linewidth=3))
rh_sym = np.array(norm_durs) @ M
ax.scatter(*rh_sym.T, s=8, color='k', alpha=.4)
ax.axis('off')

## TODO:
1. ~How to know how many gaussians to mix? We'll have to try a couple and compare using BIC~ DONE!

1. ~How to combine across clips to get a final estimation. I think we can borrow again from jacoby here and generate a mixture of the mixtures but maybe it will be a bit much... let's see.~ DONE!

## How many gaussians to mix?

Iterate over a number of gaussians (8?) and see which one gives the best BIC.

In [ ]:
maxcomps = 22
# Use bic or aic as criterion for model selection and chooose the one with the lowest value
bic = list()
for nC in range(1, maxcomps+1):
    print(nC, end='\r')
    mixt = GaussianMixture(n_components=nC, covariance_type='spherical', n_init=200, reg_covar=1e-6)
    mixt.fit(norm_durs)
    # collect one of the two
    bic.append(mixt.bic(np.array(norm_durs)))
plt.bar(range(1, maxcomps+1), bic - min(bic), bottom =min(bic))

In [ ]:
# max number of gaussian components
maxcomps = 22
x = np.linspace(0.001, 1, 101)
xG, yG = np.meshgrid(x, x)
xy = np.dstack([xG, yG])
fig, axs = plt.subplots(nrows=6, ncols=5, figsize=(16, 13))
clip_model = list()
prob_all = list()
for ax, norm_durs in zip(axs.flat, all_durs):
    crit = list()
    modcheck = list()
    for nC in range(1, maxcomps+1):
        print(nC, end='\r')
        mixt = GaussianMixture(n_components=nC, covariance_type='spherical', n_init=20, reg_covar=1e-4)
        if len(norm_durs) < nC/2:
            continue
        mixt.fit(norm_durs)
        crit.append(mixt.bic(np.array(norm_durs)))
        modcheck.append(mixt)
    modsel = np.argmin(crit)
    mixt = modcheck[modsel]
    mean_sym = mixt.means_ @ M
    if mixt.covariance_type == 'tied':
        cov_sym = [M.T @ mixt.covariances_ @ M for _ in mixt.means_]
    elif mixt.covariance_type == 'full':
        cov_sym = [M.T @ cov @ M for cov in mixt.covariances_]
    elif mixt.covariance_type == 'spherical':
        cov_sym = [M.T @ (np.eye(mixt.means_.shape[1])*cov) @ M for cov in mixt.covariances_]
    elif mixt.covariance_type == 'diag':
        cov_sym = [M.T @ np.diag(cov) @ M for cov in mixt.covariances_]

    ps = [w*multivariate_normal.pdf(xy, mu, cov) for mu, cov, w in zip(mean_sym, cov_sym, mixt.weights_)]
    prob = np.mean(ps, axis=0)
    prob[is_outside_triangle(xy, M)] = np.nan
    prob /= np.nansum(prob)
    ax.pcolormesh(x, y, prob, cmap=plt.cm.Reds)
    ax.add_patch(Polygon(rh_lims, facecolor='none', edgecolor='k', linewidth=3))
    rh_sym = np.array(norm_durs) @ M
    ax.scatter(*rh_sym.T, s=8, color='k', alpha=.4)
    ax.axis('off')
    ax.set(title=f"{modsel+1} comps")
    clip_model.append(mixt)
    prob_all.append(prob)

In [ ]:
fig, ax = plt.subplots()
ax.pcolormesh(x, y, np.mean(prob_all, axis=0), cmap=plt.cm.Reds)
ax.add_patch(Polygon(rh_lims, facecolor='none', edgecolor='k', linewidth=3))
anchors = np.array([(1, 1, 1), (2, 1, 1), (1, 2, 1),
                    (1, 1, 2), (2, 2, 1), (2, 1, 2),
                    (1, 2, 2), (1, 2, 3), (3, 1, 2),
                    (2, 3, 1), (1, 3, 2), (2, 1, 3),
                    (3, 2, 1), (3, 1, 1), (1, 3, 1),
                    (1, 1, 3), (3, 3, 1), (3, 1, 3),
                    (1, 3, 3), (3, 2, 2), (2, 3, 2),
                    (2, 2, 3), (3, 3, 2), (3, 2, 3),
                    (2, 3, 3)], dtype=float)
ach_sym = (anchors/np.sum(anchors, axis=1, keepdims=True)) @ M
ax.scatter(*ach_sym.T, marker='.', color='k', s=3)
for xy, anchor in zip(ach_sym, anchors):
    ax.annotate(str(anchor), xy, fontsize=6)
ax.axis('off')

### Future considerations

I think there is probably a better way of combining the distributions across clips.
Probably by combining gaussian components across clips based on how they overlap, giving more weight to gaussians based on more data.

I'm also interested in how covariance is estimated.
The sklearn options are limited.
I expect the noisiness of people's temporal patterns to be relatively spherical and similar across the space but I can't choose both of these options.
I also expect them to be similar across clips but I can't figure out yet how to fit all of the clips at once.
Probably have to do it by hand.

This will do for now though.

## Next

We need to be able to compare probability distributions across people for example.
Jacoby used JSD as a metric. It uses KL Divergence to compare probability distributions.
KL is not a distance measure, $D(A, B) \neq D(B, A)$.
Jacoby forces it to be transitive (or associative or whatever...) using $JSD = .5(D(A, M) + .5(D(B, M))$ where M is the mean of A and B.

We can calculate KL Divergence using the `scipy.stats.entropy` function.
I'll start by testing it across clips but the ultimate goal will be to test distance between subjects and compare to their distance in genre preferences.

In [ ]:
from scipy.stats import entropy
clips = (6, 0)

def JSD(p1, p2):
    # make sure nans are in the same place
    isna1 = np.isnan(p1)
    isna2 = np.isnan(p2)
    assert np.all(isna1 == isna2)
    # remove nans
    p1 = p1[~isna1]
    p2 = p2[~isna2]
    # run calculation
    M = (p1 + p2)/2.
    return (entropy(p1, M) + entropy(p2, M))/2

print(JSD(prob_all[clips[0]], prob_all[clips[1]]))

jsds = np.full((len(prob_all), len(prob_all)), np.nan)
for clip1 in range(len(prob_all)):
    for clip2 in range(len(prob_all)):
        jsds[clip1, clip2] = JSD(prob_all[clip1], prob_all[clip2])

plt.pcolormesh(jsds)

In [ ]:
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
jsd_cond = squareform(jsds)
Z = hierarchy.linkage(jsd_cond, method='average', optimal_ordering=True)
test = hierarchy.dendrogram(Z)

idx = np.array([int(x) for x in test['ivl']])
plt.figure()
plt.pcolormesh(jsds[idx][:, idx])